In [1]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
#from transformers import BitsAndBytesConfig
import spacy
import torch
from spacy.language import Language
from spacy import displacy
import time
import glob
import re
import math
import statistics
import os
import json
import calendar
import holidays
from pathlib import Path
from datetime import date
from datetime import datetime
import pandas as pd
import numpy as np
import collections
import hashlib
from dateutil.parser import parse
import shutil
import ast
from io import StringIO
import requests
import glob
import os

In [2]:
alias_file = "../../Summary/OTHER/aliases.json"
if os.path.exists(alias_file):
    with open(alias_file, 'r') as f:
        alias = json.load(f)
print(alias)

{'MAUS': 'MONTHLY ACTIVE USERS', 'ARR': 'ANNUAL RECURRING REVENUE', 'ARPU': 'ACTIVE REVENUE PER USER', 'ANNUAL RECURRING REVENUE ARR': 'ANNUAL RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE': 'ANNUAL RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE ARR': 'ANNUAL RECURRING REVENUE', 'NET NEW ARR': 'NET NEW ANNUAL RECURRING REVENUE', 'NET DOLLAR EXPANSION CUSTOMERS WITH MORE THAN 10 EMPLOYEES': 'NET DOLLAR EXPANSION', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 TTM REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS WITH MORE THAN $100000 OF ARR': 'LARGE PAID CUSTOMERS', 'CUSTOMERS CONTRIBUTING MORE THAN $100,000 IN REVENUE': 'LARGE PAID CUSTOMERS', 'CUSTOMERS WITH MORE THAN 10 EMPLOYEES': 'TOTAL NUMBER OF PAID CUSTOMERS', 'BUSINESSES': 'TOTAL NUMBER OF PAID CUSTOMERS', 'NET NEW SUBSCRIPTION CUSTOMERS': 'NEW PAID CUSTOMERS', 'SUBSCRIPTION CUSTOMERS': 'TOTAL NUMBER OF PAID CUSTOMERS', 'CUSTOMER COUNT': 'TOTAL NUMBER 

In [3]:
reverse_alias = dict()
for key in alias.keys():
    val = alias[key]
    if(val not in reverse_alias):
        reverse_alias[val] = list()
    reverse_alias[val].append(key)
print(reverse_alias)

{'MONTHLY ACTIVE USERS': ['MAUS', 'MAUS-GLOBAL', 'GLOBAL MONTHLY ACTIVE USERS MAUS', 'GLOBAL MONTHLY ACTIVE USERS'], 'ANNUAL RECURRING REVENUE': ['ARR', 'ANNUAL RECURRING REVENUE ARR', 'ANNUALIZED RECURRING REVENUE', 'ANNUALIZED RECURRING REVENUE ARR', 'ANNUAL RUN-RATE REVENUE ARR', 'ANNUALIZED EXIT MONTHLY RECURRING SUBSCRIPTIONS ARR', 'RINGCENTRAL TOTAL ARR', 'ENDING ARR'], 'ACTIVE REVENUE PER USER': ['ARPU', 'AVERAGE REVENUE PER CUSTOMER ARPU', 'AVERAGE REVENUE PER CUSTOMER', 'ARPU-GLOBAL'], 'NET NEW ANNUAL RECURRING REVENUE': ['NET NEW ARR'], 'NET DOLLAR EXPANSION': ['NET DOLLAR EXPANSION CUSTOMERS WITH MORE THAN 10 EMPLOYEES', 'NET DOLLAR EXPANSION CUSTOMERS WITH GREATER THAN 10 EMPLOYEES', 'REVENUE RETENTION', 'NET DOLLAR EXPANSION TOTAL NUMBER OF CUSTOMERS', 'RETENTION RATE', 'SUBSCRIPTION REVENUE RETENTION RATE', 'NET REVENUE RETENTION', 'NET DOLLAR - BASED RETENTION RATE', 'SUBSCRIPTION REVENUE NET DOLLAR EXPANSION', 'NET DOLLAR RETENTION NDR', 'NET DOLLAR RETENTION', 'NET RET

In [4]:
def getOrgData(org):
    orgDataPath = "../../Summary/orgData/"+org+".txt"
    file = Path(orgDataPath)
    if file.is_file():
        #print(True)
        with open(orgDataPath) as f:
            data = json.load(f)
        #print(data)
        return data
    return None

In [5]:
def getOrgAttr(orgData, attr):
    if not orgData:
        return None
    asplit = attr.split("|")
    parent = asplit[0]
    if parent in orgData and "SOURCE" in orgData[parent]:
        src = orgData[parent]["SOURCE"]
        if src == "YH" or (parent == "ORGPROFILE" and src == "AD"):
            p = orgData
            for i in range(0, len(asplit)):
                if asplit[i] not in p:
                    return None
                p = p[asplit[i]]
            #print(p)
            return(p)
    return None

In [6]:
def getPrevQtr(qstr):
    if not qstr:
        return None
    prvQtr = None
    qs = qstr.split("-")[0]
    year = qstr.split("-")[1]
    if(qs == "Q1"):
        year = (int(year) - 1)
        prvQtr = "Q4-"+str(year)
    elif(qs == "Q2"):
        prvQtr = "Q1-"+str(year)
    elif(qs == "Q3"):
        prvQtr = "Q2-"+str(year)
    elif(qs == "Q4"):
        prvQtr = "Q3-"+str(year)
    return(prvQtr)

In [7]:
def getEntAttr(entData, attr):
    if not entData:
        return None
    asplit = attr.split("|")
    
    p = entData
    for i in range(0, len(asplit)):
        if asplit[i] not in p:
            return None
        p = p[asplit[i]]
    #print(p)
    return(p)

In [8]:
def getAttr(allEntities, source, attrList):
    if source not in allEntities:
        return None
    p = allEntities[source]
    if not attrList:
        return p
    asplit = attrList.split("|")
    for i in range(0, len(asplit)):
        if asplit[i] not in p:
            return None
        p = p[asplit[i]]
        #print(p)
    return(p)
    #return None

In [10]:
def isMetricPresent(source, metric):
    if metric not in source:
        return False
    if metric in source and "CONFLICT" in source[metric] and source[metric]["CONFLICT"]:
        return False
    return True

In [34]:
qa = [
        "List all positive facts of Appian from Quarter Q2 Year 2025",
        "What was the revenue of Nvidia for Q2 2026"
]
print(qa)

qargs = dict()
qargs["ARGS"] = list()
qargs["ARGS"].append("KEY:POSITIVE FACTS!!TYPE:S!!ORG:APPIAN!!QTR:Q2!!YEAR:2025")
qargs["ARGS"].append("KEY:REVENUE!!TYPE:S!!ORG:NVIDIA!!QTR:Q2!!YEAR:2026")

print(json.dumps(qargs))

['List all positive facts of Appian from Quarter Q2 Year 2025', 'What was the revenue of Nvidia for Q2 2026']
{"ARGS": ["KEY:POSITIVE FACTS!!TYPE:S!!ORG:APPIAN!!QTR:Q2!!YEAR:2025", "KEY:REVENUE!!TYPE:S!!ORG:NVIDIA!!QTR:Q2!!YEAR:2026"]}
